# API Data Collection and Prediction Pipeline

In previous notebooks, we developed and evaluated regression models for predicting bike-sharing demand.

In this stage, we transition from static datasets to a dynamic pipeline that:

- Retrieves real-time weather forecast data from an external API
- Transforms the data into model-compatible features
- Generates bike demand predictions using a trained regression model

This notebook mirrors the backend logic implemented in the Shiny application, specifically:

- `model_prediction.R`
- `generate_city_weather_bike_data()`

The objective is to make the prediction pipeline transparent and reproducible.

## Import Required Libraries

We use:

- `httr`: API requests  
- `jsonlite`: JSON parsing  
- `dplyr`: data manipulation  
- `readr`: reading CSV files  

In [ ]:
# Load required libraries

# httr for API requests
library(httr)

# jsonlite for parsing JSON responses
library(jsonlite)

# dplyr for data manipulation
library(dplyr)

# readr for reading CSV files
library(readr)

## Understanding the Prediction Pipeline

The prediction pipeline consists of the following steps:

1. Retrieve weather forecast data from OpenWeather API  
2. Extract relevant features:
   - Temperature  
   - Humidity  
   - Wind Speed  
   - Visibility  
   - Hour  
   - Season  
3. Load trained regression model coefficients  
4. Apply prediction formula  
5. Classify predictions into demand levels  

This pipeline is implemented in `model_prediction.R` and used by the Shiny dashboard.

## Load Selected Cities

The cities used for prediction are stored in a CSV file.

In [ ]:
# Load selected cities dataset

cities_df <- read_csv("selected_cities.csv")

# Display cities
head(cities_df)

## Retrieve Weather Forecast Data from API

In [ ]:
# Call function from model_prediction.R

# This function retrieves 5-day forecast data for each city
weather_data <- get_weather_forecaset_by_cities(cities_df$CITY_ASCII)

# Inspect data
head(weather_data)

## Extract and Understand Features

The API provides the following key predictors:

- TEMPERATURE  
- HUMIDITY  
- WIND_SPEED  
- VISIBILITY  
- SEASONS  
- HOURS  

These features are required inputs for the regression model.

In [ ]:
# Check structure of weather dataset

str(weather_data)

## Load Regression Model Coefficients

The trained regression model is stored as coefficients in a CSV file.

In [ ]:
# Load saved regression model

model_coefs <- load_saved_model("model.csv")

# View coefficients
model_coefs

## Generate Bike Demand Predictions

We apply the regression model to weather data to predict bike demand.

In [ ]:
# Generate predictions

weather_data <- weather_data %>%
  mutate(
    BIKE_PREDICTION = predict_bike_demand(
      TEMPERATURE,
      HUMIDITY,
      WIND_SPEED,
      VISIBILITY,
      SEASONS,
      HOURS
    )
  )

# View predictions
head(weather_data)

## Classify Prediction Levels

Predictions are categorized into:

- small  
- medium  
- large  

This is used for visualization in the dashboard.

In [ ]:
# Assign prediction levels

weather_data <- weather_data %>%
  mutate(
    BIKE_PREDICTION_LEVEL = calculate_bike_prediction_level(BIKE_PREDICTION)
  )

# Inspect results
head(weather_data)

## Merge with City Coordinates

To display results on a map, we combine predictions with city latitude and longitude.

In [ ]:
# Merge weather predictions with city coordinates

final_data <- cities_df %>%
  left_join(weather_data)

# Select relevant columns
final_data <- final_data %>%
  select(
    CITY_ASCII,
    LAT,
    LNG,
    TEMPERATURE,
    HUMIDITY,
    BIKE_PREDICTION,
    BIKE_PREDICTION_LEVEL,
    LABEL,
    DETAILED_LABEL,
    FORECASTDATETIME
  )

# Display final dataset
head(final_data)

## Summary

In this notebook, we implemented a real-time prediction pipeline:

- Retrieved weather data from an API
- Transformed raw API data into model features
- Applied a trained regression model
- Generated demand predictions
- Classified predictions for visualization

This pipeline directly powers the Shiny dashboard backend.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on building a real-time prediction pipeline by integrating API data with a trained regression model.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on API integration and predictive modeling.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the transition from model development to real-time prediction:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Model Development  
- Model Evaluation  
- Model Selection  
- **API Integration & Prediction Pipeline**  
- Deployment (R Shiny Dashboard)

---

## Notes

This pipeline forms the backbone of the interactive dashboard, enabling real-time demand prediction based on weather forecasts.

---